# Lab02 — Dự đoán một ảnh: thời gian và RAM/VRAM

Model được nạp sẵn; batch 1, FP32, GPU CUDA. Mỗi model chạy trong process riêng. Tổng thời gian từ đọc ảnh đến chuỗi JSON; từng bước và bộ nhớ dùng các lượt đo riêng. Không huấn luyện lại.

Gắn Dataset private chứa gói benchmark đã kiểm tra. Không Run All notebook training cũ. RSS là peak quan sát qua lấy mẫu; CUDA allocated/reserved không phải toàn bộ VRAM process.

In [1]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, uuid, zipfile
from IPython.display import display, FileLink
matches = list(Path("/kaggle/input").rglob("package_manifest.json"))
if not matches:
    archives = list(Path("/kaggle/input").rglob("lab02-single-image-benchmark.zip"))
    if len(archives) != 1: raise RuntimeError("Attach exactly one private benchmark package")
    WORK = Path("/kaggle/working") / ("single-image-" + uuid.uuid4().hex[:8])
    WORK.mkdir()
    with zipfile.ZipFile(archives[0]) as archive:
        if any(Path(name).is_absolute() or ".." in Path(name).parts for name in archive.namelist()): raise ValueError("Unsafe archive path")
        archive.extractall(WORK)
    LAB = WORK / "lab02"
else:
    if len(matches) != 1: raise RuntimeError("Attach exactly one benchmark package")
    WORK = Path("/kaggle/working") / ("single-image-" + uuid.uuid4().hex[:8])
    WORK.mkdir()
    LAB = WORK / "lab02"
    shutil.copytree(matches[0].parent, LAB)
manifest = json.loads((LAB / "package_manifest.json").read_text())
for name, item in manifest["files"].items():
    if Path(name).is_absolute() or ".." in Path(name).parts: raise ValueError("Unsafe manifest path")
    path = WORK / name
    if hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]: raise ValueError("Checksum mismatch: " + name)
print("Verified package:", manifest["training_run"])
print("Input:", json.loads((LAB / "input.json").read_text()))

Verified package: 20261030T052121Z_cifar10_scheduled64_b895bfc1
Input: {'dataset': 'CIFAR10', 'split': 'test', 'index': 0, 'actual': 'cat', 'original_size': [32, 32], 'png_sha256': 'bc1876dae20d9c63afe538f3f5893a4e552ead5a06b677b09a0da1e6dbb83b13', 'input_policy': 'PNG exported in a separate process before loading benchmark models'}


In [2]:
import os
env = dict(os.environ, MPLBACKEND="Agg")
RESULTS = WORK / "results"
for name in ("resnet18", "densenet121"):
    command = [sys.executable, "-u", str(LAB / "scripts/benchmark_inference.py"),
               "run", "--checkpoint", str(LAB / "checkpoints" / (name + ".pt")),
               "--image", str(LAB / "input.png"), "--device", "cuda",
               "--output-dir", str(RESULTS)]
    subprocess.run(command, cwd=LAB, env=env, check=True)

{
  "run": "/kaggle/working/single-image-9baebac4/results/20261102T184034Z_single_image_resnet18_a652f073",
  "model": "resnet18",
  "first_request_ms": 6175.539883,
  "warmed_latency_ms": {
    "count": 100,
    "mean": 4.30577743,
    "median": 3.761409,
    "p95": 5.1354513
  },
  "memory": {
    "after_load": {
      "rss_bytes": 986406912,
      "allocated_bytes": 44781056,
      "reserved_bytes": 67108864
    },
    "after_warmup": {
      "rss_bytes": 1347239936,
      "allocated_bytes": 54349312,
      "reserved_bytes": 69206016
    },
    "before_pass": {
      "rss_bytes": 1347276800,
      "allocated_bytes": 54349312,
      "reserved_bytes": 69206016
    },
    "after_pass": {
      "rss_bytes": 1347346432,
      "allocated_bytes": 54349312,
      "reserved_bytes": 69206016
    },
    "peak_rss_bytes": 1347350528,
    "rss_increase_bytes": 73728,
    "peak_allocated_bytes": 59280896,
    "peak_reserved_bytes": 69206016,
    "allocated_increase_bytes": 4931584,
    "reserved_

In [3]:
import pandas as pd
rows, stage_rows = [], []
for path in sorted(RESULTS.glob("*/summary.json")):
    result = json.loads(path.read_text())
    environment = json.loads((path.parent / "environment_snapshot.json").read_text())
    memory = result["memory"]
    rows.append(dict(model=result["model"], gpu=environment["device_name"],
        first_ms=result["first_request_ms"], mean_ms=result["latency_ms"]["mean"],
        median_ms=result["latency_ms"]["median"], p95_ms=result["latency_ms"]["p95"],
        ram_loaded_mib=memory["after_load"]["rss_bytes"]/2**20,
        ram_peak_mib=memory["peak_rss_bytes"]/2**20,
        ram_increase_mib=memory["rss_increase_bytes"]/2**20,
        vram_allocated_peak_mib=memory["peak_allocated_bytes"]/2**20,
        vram_reserved_peak_mib=memory["peak_reserved_bytes"]/2**20,
        vram_increase_mib=memory["allocated_increase_bytes"]/2**20,
        predicted=result["prediction"]["predicted"], probability=result["prediction"]["probability"]))
    stage_rows.extend(dict(model=result["model"], stage=stage, **values)
                      for stage,values in result["stages_ms"].items())
if len(rows) != 2: raise RuntimeError("Expected two completed model benchmarks")
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULTS / "comparison.csv", index=False)
display(comparison.round(4))
display(pd.DataFrame(stage_rows).round(4))
print("Timing stages are diagnostic; their sum is not the independently measured total.")

,model,gpu,first_ms,mean_ms,median_ms,p95_ms,ram_loaded_mib,ram_peak_mib,ram_increase_mib,vram_allocated_peak_mib,vram_reserved_peak_mib,vram_increase_mib,predicted,probability
0,resnet18,Tesla T4,6175.5399,4.3058,3.7614,5.1355,940.7109,1284.9336,0.0703,56.5347,66.0,4.7031,cat,0.9617
1,densenet121,Tesla T4,477.5517,15.5548,15.3290,16.1566,986.8008,1309.5156,0.0742,42.8477,48.0,6.7344,cat,0.9966


,model,stage,count,mean,median,p95
0,resnet18,read,20,0.2307,0.2219,0.2792
1,resnet18,preprocess,20,0.3288,0.3281,0.3764
2,resnet18,transfer,20,0.0680,0.0667,0.0774
3,resnet18,forward,20,2.8613,2.8227,3.1349
4,resnet18,postprocess,20,0.1141,0.1117,0.1276
5,resnet18,serialize,20,0.0467,0.0453,0.0560
6,densenet121,read,20,0.2295,0.2263,0.2528
7,densenet121,preprocess,20,0.3482,0.3308,0.4097
8,densenet121,transfer,20,0.0705,0.0688,0.0818
9,densenet121,forward,20,14.5822,14.5224,15.1421


Timing stages are diagnostic; their sum is not the independently measured total.


In [4]:
archive = WORK / "single-image-results.zip"
if archive.exists(): raise FileExistsError(archive)
shutil.make_archive(str(archive.with_suffix("")), "zip", root_dir=RESULTS)
print("Results:", archive)
display(FileLink(str(archive)))

Results: /kaggle/working/single-image-9baebac4/single-image-results.zip


/kaggle/working/single-image-9baebac4/single-image-results.zip